# Imitation Learning: BC & DAgger

Wiki reference for [imitation learning](https://ml-viz-ruby.vercel.app/wiki/imitation-learning).

**The idea in one sentence.** Behavioral Cloning (BC) trains a policy on expert
state-action pairs, but the expert only visits a **narrow slice** of states — so when the
learner drifts into unseen states its errors compound (**covariate shift**); **DAgger** fixes
this by rolling out the *learner*, then querying the expert to label those new states.

We implement the expert, BC, and DAgger from scratch, **validate that the expert converges and
that BC recovers its gain**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.edgecolor': '#2d3748',
    'grid.color': '#2d3748',
    'axes.grid': True,
})

np.random.seed(42)

## 1. The environment: 1D reaching task

State: position $s \in [-1, 1]$. Goal: position 0 (the center). Action: velocity $a \in [-1, 1]$. Next state: $s' = \text{clip}(s + 0.1a, -1, 1)$. Expert policy: $a^* = -10s$ (proportional control pointing toward center).

In [ ]:
TARGET = 0.0
DT = 0.1

def step(s, a):
    return np.clip(s + DT * a, -1.0, 1.0)

def expert_policy(s):
    """P controller: move proportionally toward target."""
    return np.clip(-10 * (s - TARGET), -1, 1)

def rollout(policy_fn, s0, T=30):
    s = s0
    states = [s]
    for _ in range(T):
        a = policy_fn(s)
        s = step(s, a)
        states.append(s)
    return np.array(states)

# Demonstrate expert on several starting positions
s0s = [-0.9, -0.5, 0.0, 0.5, 0.9]
fig, ax = plt.subplots(figsize=(10, 4))
for s0 in s0s:
    traj = rollout(expert_policy, s0)
    ax.plot(traj, linewidth=2, alpha=0.8)
ax.axhline(TARGET, color='white', linestyle='--', linewidth=1.5, label='Target')
ax.set_xlabel('Timestep')
ax.set_ylabel('Position')
ax.set_title('Expert Policy Trajectories', color='#e2e8f0')
ax.legend()
plt.tight_layout()
plt.show()

### Validate: the expert drives every start to the target

The expert is a proportional controller ($a = -10\,s$, clipped), so from *any* starting
position it steers the state to the target. We confirm it converges from the extremes.

In [ ]:
for s0 in [-0.9, 0.9]:
    end = rollout(expert_policy, s0, T=30)[-1]
    print(f'expert from s0={s0:+.1f} ends at {end:.4f}')
    assert abs(end - TARGET) < 0.01, 'the expert P-controller drives every start to the target'
print('\n✅ the expert is a competent policy we want to imitate')

## 2. Behavioral Cloning — distribution shift failure

In [ ]:
# Collect expert demonstrations from near-target starting positions only
n_demos = 50
expert_states = np.random.uniform(-0.2, 0.2, n_demos)  # expert starts near center
expert_data = []
for s0 in expert_states:
    traj = rollout(expert_policy, s0, T=20)
    for s in traj[:-1]:
        expert_data.append((s, expert_policy(s)))

X_bc = np.array([x for x, _ in expert_data]).reshape(-1, 1)
y_bc = np.array([a for _, a in expert_data])

# Linear BC policy: fit a = w * s
w_bc = (X_bc[:, 0] @ y_bc) / (X_bc[:, 0] @ X_bc[:, 0])

def bc_policy(s):
    return np.clip(w_bc * s, -1, 1)

# Test on FAR starting positions (out of expert distribution)
test_starts = np.random.uniform(-1.0, 1.0, 20)
T_test = 50

bc_final_errors = []
expert_final_errors = []
for s0 in test_starts:
    traj_bc = rollout(bc_policy, s0, T=T_test)
    traj_ex = rollout(expert_policy, s0, T=T_test)
    bc_final_errors.append(abs(traj_bc[-1] - TARGET))
    expert_final_errors.append(abs(traj_ex[-1] - TARGET))

print(f"Expert final error:   mean={np.mean(expert_final_errors):.4f}, max={np.max(expert_final_errors):.4f}")
print(f"BC final error:       mean={np.mean(bc_final_errors):.4f}, max={np.max(bc_final_errors):.4f}")
print(f"\nBC was trained on starts ∈ [-0.2, 0.2] but tested on [-1, 1] — distribution shift!")

# Plot trajectories
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for s0 in test_starts[:8]:
    axes[0].plot(rollout(expert_policy, s0, T=T_test), linewidth=1.5, alpha=0.7)
    axes[1].plot(rollout(bc_policy, s0, T=T_test), linewidth=1.5, alpha=0.7)
for ax, title in zip(axes, ['Expert', 'Behavioral Cloning (distribution shift)']):
    ax.axhline(TARGET, color='white', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Timestep')
    ax.set_ylabel('Position')
    ax.set_title(title, color='#e2e8f0')
plt.tight_layout()
plt.show()

## 3. DAgger — dataset aggregation fixes distribution shift

In [ ]:
def fit_linear(X, y):
    """Fit w s.t. y ≈ w * X (1D)."""
    x_flat = X[:, 0]
    return (x_flat @ y) / (x_flat @ x_flat + 1e-9)

# DAgger iterations
n_dagger_iters = 10
T_rollout = 30
n_starts_per_iter = 10

# Start with empty dataset
D_states, D_actions = [], []
dagger_errors = []
bc_errors_over_time = []

# Initial BC policy (random)
w_dagger = 0.0

for iteration in range(n_dagger_iters):
    # 1. Roll out current policy (starting from varied positions)
    starts = np.random.uniform(-1.0, 1.0, n_starts_per_iter)
    new_states = []
    for s0 in starts:
        s = s0
        for _ in range(T_rollout):
            new_states.append(s)
            a = np.clip(w_dagger * s, -1, 1)  # current policy
            s = step(s, a)

    # 2. Label with expert
    new_actions = [expert_policy(s) for s in new_states]

    # 3. Aggregate
    D_states.extend(new_states)
    D_actions.extend(new_actions)

    # 4. Refit policy on aggregated dataset
    X_agg = np.array(D_states).reshape(-1, 1)
    y_agg = np.array(D_actions)
    w_dagger = fit_linear(X_agg, y_agg)

    # Evaluate
    errs = [abs(rollout(lambda s: np.clip(w_dagger*s, -1, 1), s0, T=T_test)[-1] - TARGET)
            for s0 in test_starts]
    dagger_errors.append(np.mean(errs))
    bc_errors_over_time.append(np.mean(bc_final_errors))  # BC doesn't improve

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(dagger_errors, 'o-', color='#10b981', linewidth=2, label='DAgger')
ax1.axhline(np.mean(bc_final_errors), color='#f43f5e', linestyle='--', linewidth=2, label='BC (fixed)')
ax1.axhline(np.mean(expert_final_errors), color='white', linestyle=':', linewidth=1, label='Expert')
ax1.set_xlabel('DAgger iteration')
ax1.set_ylabel('Mean final position error')
ax1.set_title('DAgger Converges; BC Is Stuck', color='#e2e8f0')
ax1.legend()

# Show DAgger trajectories after final iteration
w_final = w_dagger
for s0 in test_starts[:8]:
    ax2.plot(rollout(lambda s: np.clip(w_final*s, -1, 1), s0, T=T_test), linewidth=1.5, alpha=0.7)
ax2.axhline(TARGET, color='white', linestyle='--', linewidth=1.5)
ax2.set_xlabel('Timestep')
ax2.set_ylabel('Position')
ax2.set_title('DAgger Policy After 10 Iterations', color='#e2e8f0')
plt.tight_layout()
plt.show()

print(f"DAgger final error: {dagger_errors[-1]:.4f}")
print(f"BC error:           {np.mean(bc_final_errors):.4f}")

### Validate: BC recovers the gain and DAgger converges

BC fits the expert's linear feedback gain (negative — push back toward the target), and DAgger
reaches the target after its iterations. We confirm both.

In [ ]:
print(f'BC learned gain w = {float(w_bc):.2f} (expert uses -10); BC training states in [{X_bc.min():.2f}, {X_bc.max():.2f}]')
print(f'DAgger final error = {dagger_errors[-1]:.4f}')
assert float(w_bc) < -5, 'BC recovers the expert negative-feedback gain'
assert dagger_errors[-1] < 0.05, 'DAgger converges to the target'
print('\n✅ both learn the task here — but they see very different STATE distributions (next cell)')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **covariate shift** | BC fails on states the expert never visited (demo motivates DAgger) |
| **compounding errors** | small BC errors snowball over a trajectory |
| **DAgger needs the expert online** | it must be queryable during training — often costly |
| **linear-toy caveat** | simple tasks let BC extrapolate, hiding the problem (verified) |
| **distribution of demos** | narrow demos = narrow competence |

Demo: DAgger's dataset covers a far wider state range than BC's.

In [ ]:
# The crux of why DAgger beats BC in general: COVERAGE. BC only ever sees the expert's narrow
# states (the expert starts near the target), so it has NO data on how to act far away. DAgger
# rolls out the LEARNER and labels the states IT visits — a much wider range — closing the
# covariate-shift gap. (On this linear task BC happens to extrapolate fine, but with a nonlinear
# expert that extrapolation fails.) We compare the state coverage of the two datasets.
bc_width = float(X_bc.max() - X_bc.min())
dagger_width = float(max(D_states) - min(D_states))
print(f'BC state coverage:     [{X_bc.min():.2f}, {X_bc.max():.2f}]  width {bc_width:.2f}')
print(f'DAgger state coverage: [{min(D_states):.2f}, {max(D_states):.2f}]  width {dagger_width:.2f}')
assert dagger_width > 2 * bc_width, 'DAgger collects the learners OWN (wider) states -> coverage where BC has none'
print('\nBC is trained where the EXPERT goes; DAgger where the LEARNER goes -> DAgger fixes covariate shift.')

## ✏️ Your turn

**Exercise 1 — BC with more data.** Does giving BC more expert data (from a wider range of starting positions) close the gap with DAgger? Collect 500 expert demonstrations starting from $s_0 \in [-1, 1]$ (full range), fit a new linear BC policy, and compare its final error to DAgger.

In [ ]:
# TODO(you): collect expert demos from the full range [-1, 1]
# and compare to BC trained on narrow range and DAgger
n_demos_wide = 500
# expert_starts_wide = np.random.uniform(-1.0, 1.0, n_demos_wide)
# ... fit w_bc_wide, evaluate errors

In [ ]:
# Assert cell
expert_starts_wide = np.random.uniform(-1.0, 1.0, 500)
wide_data = []
for s0 in expert_starts_wide:
    for s in rollout(expert_policy, s0, T=20)[:-1]:
        wide_data.append((s, expert_policy(s)))
X_wide = np.array([x for x, _ in wide_data]).reshape(-1, 1)
y_wide = np.array([a for _, a in wide_data])
w_bc_wide = fit_linear(X_wide, y_wide)
wide_errs = [abs(rollout(lambda s: np.clip(w_bc_wide*s, -1, 1), s0, T=T_test)[-1] - TARGET)
             for s0 in test_starts]
print(f"BC (narrow demos) error:  {np.mean(bc_final_errors):.4f}")
print(f"BC (wide demos) error:    {np.mean(wide_errs):.4f}")
print(f"DAgger error:             {dagger_errors[-1]:.4f}")
assert np.mean(wide_errs) < 0.05 and np.mean(bc_final_errors) < 0.05, "both narrow- and wide-demo BC solve this LINEAR task (BC extrapolates a linear expert), so wider demos do not help here"

<details><summary>Solution</summary>

```python
expert_starts_wide = np.random.uniform(-1.0, 1.0, 500)
wide_data = []
for s0 in expert_starts_wide:
    for s in rollout(expert_policy, s0, T=20)[:-1]:
        wide_data.append((s, expert_policy(s)))

X_wide = np.array([x for x, _ in wide_data]).reshape(-1, 1)
y_wide = np.array([a for _, a in wide_data])
w_bc_wide = fit_linear(X_wide, y_wide)

wide_errs = [abs(rollout(lambda s: np.clip(w_bc_wide*s,-1,1), s0, T=T_test)[-1] - TARGET)
             for s0 in test_starts]
print(f"BC (wide) error: {np.mean(wide_errs):.4f}")
```

With full-coverage expert data, BC can match DAgger for this simple linear task. The key insight: when the expert's state distribution covers the test distribution, BC works. DAgger is needed when you *can't* collect expert data from all relevant states upfront — which is the typical case in complex tasks where the expert can't enumerate all possible states in advance.

</details>

## Key takeaways

- **BC clones expert state-action pairs** but only sees the expert's narrow state distribution.
- **Covariate shift:** off-distribution, BC's errors compound — the core BC weakness.
- **DAgger** rolls out the learner and has the expert label those states — far wider coverage
  (demo).
- **On this linear toy BC extrapolates fine** (verified) — the failure appears with complex,
  nonlinear experts.